## Tracing & Debugging a Walmart Pickup Flow

In [1]:
# Tracing and Debugging with OpenTelemetry.

In [2]:
# Scenario 1: Successful pickup (WM-104)
# A customer asks whether order WM-104 is ready.

# The system successfully retrieves:
#     - Order status: Ready
#     - Pickup bay: 3

# Customer response:
#     "Your order is ready. Park in bay 3."

In [3]:
# Scenario 2: Failed pickup lookup (WM-106)

# The lookup service experiences a simulated timeout, so the system cannot verify the order.

# Customer response:
#     "I could not verify your pickup details. Please ask a pickup associate."

In [4]:
# - Trace: The complete journey of one customer request.
# - Span: An individual operation performed during that journey.

## First, picture the request

A **trace** is the whole journey for one request. A **span** is one timed step in that journey. Parent-child links tell us which steps belong together.

```text
pickup.request                 one customer request (root span)
├── pickup.lookup              check the pickup order
└── pickup.reply               draft the customer answer
```

If the lookup times out, should `pickup.reply` appear? Pause and make a prediction before running the trace.

We use the same Walmart policy as Notebook 6: report only verified pickup facts; when lookup fails, be clear and direct the customer to an associate.


In [ ]:
# What is a span?

# A span represents one operation being performed by an application.

# For example:
# - pickup.request — Customer submits an order inquiry.
# - pickup.lookup — System checks order details.
# - pickup.reply — System prepares the response.
# Each operation can have its own span, allowing us to understand what happened at every stage.

In [5]:
# Import the OpenTelemetry SDK pieces used to create and inspect spans.
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import Status, StatusCode

# Keep finished spans in memory so the class can inspect them immediately.
exporter = InMemorySpanExporter()

# Create a provider that sends each finished span to the in-memory exporter.
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))

# Get a tracer for this synthetic Walmart pickup service.
tracer = provider.get_tracer("walmart.pickup.training")

# Confirm that the local trace recorder is ready.
print("Local trace recorder ready.")


Local trace recorder ready.


**Production translation** · This lesson keeps spans in memory. A deployed service commonly exports them through OTLP to an OpenTelemetry Collector and an approved observability backend; production also configures context propagation, sampling, retention, and access controls. See the [Collector documentation](https://opentelemetry.io/docs/collector/).


## Step 1: understand the tiny Walmart workflow

Before adding tracing, read the business logic. `lookup_pickup` returns known synthetic evidence, or raises a timeout when we ask it to simulate an outage. `create_customer_reply` uses the evidence to produce a safe reply.

The order records and timeout are classroom fixtures. No Walmart system is contacted.


In [ ]:
# Store only the fictional ready order used in this lesson.
pickup_orders = {
    "WM-104": {"status": "ready", "bay": 3},
}

# Define the safe response for an order that cannot be verified.
safe_handoff = "I could not verify your pickup details. Please ask a pickup associate."

# Simulate the order lookup that a real service would perform.
def lookup_pickup(order_id, *, simulate_timeout=False): # lookup_pickup("WM-104", simulate_timeout=True)
    # Raise a predictable error when this exercise requests an outage.
    if simulate_timeout:
        raise TimeoutError("Simulated Walmart pickup lookup timeout")

    # Return the synthetic order record, or None when no record exists.
    return pickup_orders.get(order_id)

# Turn verified lookup evidence into a customer-facing answer.
def create_customer_reply(lookup_result): # {"status": "ready", "bay": 3} ==> Your order is ready. Park in bay 3.
    # Never invent a status or bay when no verified result is available.
    if lookup_result is None:
        return safe_handoff

    # Use the verified status and bay from the order record.
    if lookup_result["status"] == "ready":
        return f"Your order is ready. Park in bay {lookup_result['bay']}."

    # Use the safe handoff for any status this lesson does not handle.
    return safe_handoff

# Run the ordinary path once before adding tracing.
ready_record = lookup_pickup("WM-104")
print(create_customer_reply(ready_record))


Your order is ready. Park in bay 3.


## Step 2: wrap the request and its work in spans

We create one root span for the request, then child spans for the lookup and reply. The `try/except` is intentional: it preserves the safe customer response while recording the technical failure for the operator.


In [7]:
# Trace one Walmart pickup request and return its customer-facing answer.
def trace_pickup_request(case_id, order_id, *, simulate_timeout=False):
    # The root span represents the complete customer request.
    with tracer.start_as_current_span("pickup.request") as request_span:
        # Add synthetic, searchable context to the root span.
        request_span.set_attribute("walmart.case_id", case_id)
        request_span.set_attribute("walmart.workflow", "pickup")

        try:
            # The child span measures the order lookup step.
            with tracer.start_as_current_span("pickup.lookup") as lookup_span:
                # This ID is synthetic; never put real customer data in a span.
                lookup_span.set_attribute("walmart.order_id", order_id)

                # Call the same lookup function used in the untraced example.
                result = lookup_pickup(
                    order_id,
                    simulate_timeout=simulate_timeout,
                )

                # Record the business result so the trace explains the path.
                lookup_status = result["status"] if result else "not_found"
                lookup_span.set_attribute("pickup.lookup.status", lookup_status)

            # The reply span exists only after lookup completes successfully.
            with tracer.start_as_current_span("pickup.reply") as reply_span:
                answer = create_customer_reply(result)
                reply_span.set_attribute("pickup.reply.created", True)

            # Mark a normal response separately from a safe handoff.
            outcome = "answered" if result else "handoff"
            request_span.set_attribute("pickup.outcome", outcome)
            return answer

        except TimeoutError as error:
            # Attach the exception to the request and mark its technical status.
            request_span.record_exception(error)
            request_span.set_status(Status(StatusCode.ERROR))
            request_span.set_attribute("error.type", type(error).__name__)

            # Keep the customer safe even though the lookup failed.
            request_span.set_attribute("pickup.outcome", "handoff")
            return safe_handoff


## Step 3: run success and failure

The two calls below differ by one condition: WM-106 deliberately times out during lookup. Compare the customer replies first; then inspect the spans to locate the difference.


In [8]:
# Clear any earlier classroom spans so this comparison is easy to read.
exporter.clear()

# Run the successful WM-104 pickup lookup.
success_reply = trace_pickup_request("WM-104", "WM-104")

# Run WM-106 with an intentionally simulated lookup timeout.
timeout_reply = trace_pickup_request(
    "WM-106",
    "WM-106",
    simulate_timeout=True,
)

# Compare what the customer receives in each case.
print("WM-104:", success_reply)
print("WM-106:", timeout_reply)

# Verify the safe behavior before inspecting the telemetry.
assert "bay 3" in success_reply.lower()
assert "associate" in timeout_reply.lower()


WM-104: Your order is ready. Park in bay 3.
WM-106: I could not verify your pickup details. Please ask a pickup associate.


## Step 4: read the trace like an incident investigator

Each trace has its own trace ID. The root span's ID lets us find its child spans; the status and exception event show where a failure occurred.

Look for the prediction from Step 1: a timeout should prevent the reply span from being created.


In [9]:
# Read all spans completed by the two synthetic requests.
finished_spans = exporter.get_finished_spans()

# Select the root request spans and index them by their Walmart case.
request_spans = [
    span for span in finished_spans
    if span.name == "pickup.request"
]
root_by_case = {
    span.attributes["walmart.case_id"]: span
    for span in request_spans
}
# eg:
# {
#     "WM-104": <root span for WM-104>,
#     "WM-106": <root span for WM-106>
# }

# Print each request and its child operations in execution order.
for request_span in request_spans:
    case_id = request_span.attributes["walmart.case_id"]
    trace_id = f"{request_span.context.trace_id:032x}"
    print(
        f"\n{case_id} | trace={trace_id} | "
        f"outcome={request_span.attributes['pickup.outcome']}"
    )

    # Keep only spans whose parent is this request.
    child_spans = [
        span for span in finished_spans
        if span.parent
        and span.parent.span_id == request_span.context.span_id
    ]

    # Sort by start time so the trace reads from first step to last.
    for child_span in sorted(child_spans, key=lambda span: span.start_time):
        print(
            f"  {child_span.name} | "
            f"{child_span.status.status_code.name}"
        )

        # Show exception text only for this synthetic classroom failure.
        for event in child_span.events:
            if event.name == "exception":
                print("    error:", event.attributes["exception.message"])

# Confirm the timeout trace has no reply span and does have a failed lookup.
success_children = [
    span.name for span in finished_spans
    if span.parent
    and span.parent.span_id == root_by_case["WM-104"].context.span_id
]
timeout_children = [
    span for span in finished_spans
    if span.parent
    and span.parent.span_id == root_by_case["WM-106"].context.span_id
]
assert "pickup.reply" in success_children
assert all(span.name != "pickup.reply" for span in timeout_children)
assert any(span.status.status_code == StatusCode.ERROR for span in timeout_children)
print("\nTrace checks passed.")



WM-104 | trace=16bb41fcac08afbfe7e073c0a95482de | outcome=answered
  pickup.lookup | UNSET
  pickup.reply | UNSET

WM-106 | trace=24c6cb6c6d8be5d923dd89d2575ec0b2 | outcome=handoff
  pickup.lookup | ERROR
    error: Simulated Walmart pickup lookup timeout

Trace checks passed.


## Step 5: connect the trace back to Notebook 6's rubric

Observability and response quality are related, but they are not the same measure. WM-106 has a **technical error** because the lookup timed out; its customer reply can still be **grounded and actionable** because it admits uncertainty and directs the customer to an associate.

For these two examples, we apply the same three ideas as Notebook 6: grounded, actionable, and critical failure.


In [10]:
# Score the two replies using the Notebook 6 business criteria.
evaluation_results = []

# Keep each case and answer together for the simple classroom checks.
case_answers = [
    ("WM-104", success_reply),
    ("WM-106", timeout_reply),
]

# Check whether each answer is grounded and gives a useful next step.
for case_id, answer in case_answers:
    text = answer.casefold()

    # WM-104 must report its verified ready status and bay 3.
    if case_id == "WM-104":
        grounded = "ready" in text and "bay 3" in text
        actionable = "bay 3" in text

    # WM-106 must disclose uncertainty and direct the customer to help.
    else:
        grounded = "could not verify" in text
        actionable = "associate" in text

    # A critical failure is an ungrounded or non-actionable reply.
    critical_failure = not grounded or not actionable

    # Add runtime health from the matching request trace.
    request_span = root_by_case[case_id]
    technical_error = request_span.status.status_code == StatusCode.ERROR

    # Save the two different kinds of evidence side by side.
    evaluation_results.append({
        "case_id": case_id,
        "technical_error": technical_error,
        "grounded": grounded,
        "actionable": actionable,
        "critical_failure": critical_failure,
    })

# Print a compact table for discussion.
print("case    tech_error  grounded  actionable  critical_failure")
for row in evaluation_results:
    print(
        f"{row['case_id']:<7} "
        f"{str(row['technical_error']):<11} "
        f"{str(row['grounded']):<10} "
        f"{str(row['actionable']):<12} "
        f"{row['critical_failure']}"
    )

# The timeout is an operational failure, not a critical customer-answer failure.
wm106_result = next(
    row for row in evaluation_results if row["case_id"] == "WM-106"
)
assert wm106_result["technical_error"] is True
assert wm106_result["critical_failure"] is False
print("Key lesson: a request can fail technically and still respond safely.")


case    tech_error  grounded  actionable  critical_failure
WM-104  False       True       True         False
WM-106  True        True       True         False
Key lesson: a request can fail technically and still respond safely.


## Optional instructor extension: trace one live model call

The core lesson does not need an API key. To demonstrate a real model span, set `RUN_LIVE = True` in the next cell. It sends one **fictional** WM-104 request to OpenAI and incurs provider usage charges; it does not call a Walmart service. Keep it off for participants who do not have approved credentials.


In [11]:
# Keep the live provider example optional for participants.
RUN_LIVE = True  # Set to False to skip the live model call and avoid API calls.

if RUN_LIVE:
    # Import live-call dependencies only when the instructor enables the example.
    import os
    from pathlib import Path
    from dotenv import load_dotenv
    from openai import OpenAI

    # Load keys from local files without replacing a kernel environment key.
    for env_file in (Path("my.env"), Path(".env")):
        if env_file.is_file():
            load_dotenv(env_file, override=False)

    # Fail clearly before the request if no key is available.
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError(
            "Set OPENAI_API_KEY in the kernel environment, my.env, or .env."
        )

    # Call the model inside a span and include only synthetic pickup evidence.
    with OpenAI(api_key=api_key, timeout=20.0, max_retries=0) as client:
        with tracer.start_as_current_span("gen_ai.reply") as model_span:
            response = client.responses.create(
                model="gpt-4.1-mini",
                input=(
                    "A fictional Walmart pickup order is ready; verified bay is 3. "
                    "Reply in one concise sentence and do not add other facts."
                ),
                store=False,
            )

            # Record model and usage metadata, not the prompt or raw reply.
            model_span.set_attribute("gen_ai.request.model", response.model)
            if response.usage is not None:
                model_span.set_attribute(
                    "gen_ai.usage.input_tokens",
                    response.usage.input_tokens,
                )
                model_span.set_attribute(
                    "gen_ai.usage.output_tokens",
                    response.usage.output_tokens,
                )

    # Show the synthetic response after the span has completed.
    print(response.output_text)
else:
    print("Live model call skipped; the core lesson is complete.")


Your Walmart pickup order is ready for pickup at bay 3.


# Happy Learning